# Lesson 6.3 — Analytics ADQL Queries in Grafana

## Learning Objectives
- Configure the Analytics query type in the Grafana AppDynamics plugin
- Write ADQL queries in Grafana's Analytics query editor with autocomplete
- Use the Limit field to control result set size
- Match ADQL query shape to the correct Grafana visualization type

**Reference**: [Analytics Query Type](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/#analytics-query-type) | [Query Examples](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/#analytics-query-examples)

## 1. Analytics Query Type Configuration

### Prerequisite
Analytics queries require separate authentication (Analytics API URL + API key + Global Account Name) configured in the datasource settings (Lesson 6.1).

### Using the Analytics Query Editor:
1. In panel edit mode, select **Analytics** from the query type radio buttons
2. A code editor appears with syntax highlighting and autocomplete
3. Type your ADQL query
4. Set the **Limit** field (default 100, max 1000)
5. Use the dashboard time picker to set the time range

### Fields:
| Field | Description |
|-------|-------------|
| **Query** | The ADQL query to execute |
| **Limit** | Max records to retrieve (default 100, max 1000). For aggregation queries, limits result rows, not raw records scanned |

## 2. Official Analytics Query Examples

These examples are taken directly from the [official Grafana AppDynamics query editor documentation](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/#analytics-query-examples):

### Count transactions grouped by name, filtered by a template variable:
```sql
SELECT distinct(transactionName), count(*) FROM transactions
WHERE transactionName IN (${transactionName:doublequote})
```

### Aggregate response times into a time series by user experience:
```sql
SELECT series(eventTimestamp, '1m'), userExperience, count(*) FROM transactions
```

### Query the slowest transactions over the selected time range:
```sql
SELECT transactionName, max(responseTime) AS maxResponseTime FROM transactions
ORDER BY maxResponseTime DESC
LIMIT 10
```

### Query application logs for errors:
```sql
SELECT eventTimestamp, MSGTY, MSGTXT, Application FROM application_logs
WHERE MSGTY = 'E'
```

### Query browser performance records:
```sql
SELECT pageUrl, avg(domReadyTime) AS avgDomReady, count(*) AS total FROM browser_records
ORDER BY avgDomReady DESC
```

In [ ]:
# Complete Grafana Analytics query library with visualization recommendations

grafana_analytics_queries = [
    {
        "use_case": "User experience over time (stacked bar/time series)",
        "panel_type": "Time Series or Bar Chart",
        "query": "SELECT series(eventTimestamp, '1m'), userExperience, count(*) FROM transactions",
        "limit": 100,
        "note": "series() produces time-bucketed data; group by userExperience for stacked view"
    },
    {
        "use_case": "Top 10 slowest transactions (table)",
        "panel_type": "Table",
        "query": "SELECT transactionName, max(responseTime) AS maxResponseTime FROM transactions ORDER BY maxResponseTime DESC LIMIT 10",
        "limit": 10,
        "note": "Table panel works best for this"
    },
    {
        "use_case": "Error rate trend (time series line chart)",
        "panel_type": "Time Series",
        "query": "SELECT series(eventTimestamp, '1m'), count(*) AS errors FROM transactions WHERE errorOccurred = true",
        "limit": 100,
        "note": ""
    },
    {
        "use_case": "Transaction count by name, filtered by template variable",
        "panel_type": "Bar Chart or Table",
        "query": "SELECT distinct(transactionName), count(*) FROM transactions WHERE transactionName IN (${transactionName:doublequote})",
        "limit": 100,
        "note": "Requires ${transactionName} template variable with :doublequote formatter"
    },
    {
        "use_case": "Identify slow transactions (for table investigation)",
        "panel_type": "Table",
        "query": "SELECT transactionName, responseTime, eventTimestamp FROM transactions WHERE responseTime > 5000 ORDER BY responseTime DESC",
        "limit": 50,
        "note": "Set Limit to 50 to see top 50 slowest transactions"
    },
    {
        "use_case": "User experience distribution (pie chart)",
        "panel_type": "Pie Chart",
        "query": "SELECT series(eventTimestamp, '5m'), userExperience, count(*) FROM transactions",
        "limit": 100,
        "note": "For a pie chart showing distribution, use without series() — just count by userExperience"
    },
    {
        "use_case": "Browser performance by page URL",
        "panel_type": "Table",
        "query": "SELECT pageUrl, avg(domReadyTime) AS avgDomReady, count(*) AS total FROM browser_records ORDER BY avgDomReady DESC",
        "limit": 100,
        "note": "References the browser_records table"
    },
    {
        "use_case": "Application error logs",
        "panel_type": "Logs or Table",
        "query": "SELECT eventTimestamp, MSGTY, MSGTXT, Application FROM application_logs WHERE MSGTY = 'E'",
        "limit": 200,
        "note": "Recent error log messages; use Logs panel if you want log-style display"
    },
]

print("=== Grafana AppDynamics Analytics Query Library ===")
print("Reference: https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/")
print()
for i, q in enumerate(grafana_analytics_queries, 1):
    print(f"{i}. [{q['panel_type'].upper()}] {q['use_case']}")
    print(f"   Query: {q['query']}")
    print(f"   Limit: {q['limit']}")
    if q['note']:
        print(f"   Note : {q['note']}")
    print()

In [ ]:
# EXERCISE 6.3: Match ADQL queries to Grafana panel types
#
# For each ADQL query below, choose the best Grafana visualization type
# and explain why.

queries_to_match = [
    {
        "query": "SELECT series(eventTimestamp, '1m'), count(*) FROM transactions WHERE errorOccurred = true",
        "best_panel": "???",   # TODO: Time Series / Table / Stat / Pie
        "reason": "???"
    },
    {
        "query": "SELECT transactionName, count(*) AS total, avg(responseTime) FROM transactions ORDER BY total DESC LIMIT 10",
        "best_panel": "???",
        "reason": "???"
    },
    {
        "query": "SELECT count(*) FROM transactions WHERE errorOccurred = true",
        "best_panel": "???",
        "reason": "???"
    },
    {
        "query": "SELECT country, count(*) FROM transactions WHERE errorOccurred = true ORDER BY count(*) DESC",
        "best_panel": "???",
        "reason": "???"
    },
    {
        "query": "SELECT series(eventTimestamp, '5m'), userExperience, count(*) FROM transactions",
        "best_panel": "???",
        "reason": "???"
    },
]

print("Panel Type Matching Exercise:")
for i, q in enumerate(queries_to_match, 1):
    print(f"\n{i}. Query: {q['query'][:80]}..." if len(q['query']) > 80 else f"\n{i}. Query: {q['query']}")
    print(f"   Best panel: {q['best_panel']}")
    print(f"   Reason    : {q['reason']}")

## Summary

- Analytics query type uses ADQL to query the AppDynamics **Events Service** (not the Metrics Store)
- The **Limit** field controls how many records or aggregation rows are returned
- `series()` function produces time-series data for **Time Series** panels in Grafana
- Without `series()`, queries return tabular data — use **Table**, **Stat**, or **Pie** panels
- Use `${var:doublequote}` formatting for multi-select Grafana template variables in ADQL `IN` clauses

---

**Next**: [Lesson 6.4 — Health, Events, Tiers, and Metric Names Query Types](lesson-6.4-health-events-tiers-queries.ipynb)